In [4]:
!gdown 1s2nKQYycgfXn8P0AybfDyoB2xx2bgS1m -O data/passive_subjects.csv

Downloading...
From: https://drive.google.com/uc?id=1s2nKQYycgfXn8P0AybfDyoB2xx2bgS1m
To: /Users/camilarojasguajardo/Desktop/Magister/lobby-recsys/data/passive_subjects.csv
100%|██████████████████████████████████████| 1.65M/1.65M [00:00<00:00, 19.0MB/s]


In [2]:
import os
import pandas as pd
import numpy as np
from src.utils import _norm_text, _pick_org
from src.state_of_art_models.lightGCN_base import LightGCNRunner

# Format data

In [3]:
def build_lightgcn_inputs(active, aud, passive):
    active = active.copy()
    aud = aud.copy()
    passive = passive.copy()

    active["display_name"] = active["Nombre completo"].fillna("").astype(str).str.strip()
    active["display_org"]  = active.apply(_pick_org, axis=1).astype(str).str.strip()
    active["__name_norm"]  = active["display_name"].map(_norm_text)
    active["__org_norm"]   = active["display_org"].map(_norm_text)
    active["user_key"]     = (active["__name_norm"] + " | " + active["__org_norm"]).str.strip(" |")

    aud["timestamp"] = pd.to_datetime(aud["fecha"], errors="coerce")

    aud_cols = ["audiencia_id", "sujeto_pasivo_id", "institucion_id", "timestamp"]
    edges = active.merge(aud[aud_cols], on="audiencia_id", how="inner", suffixes=("_act", "_aud"))
    sp_col  = "sujeto_pasivo_id_aud" if "sujeto_pasivo_id_aud" in edges.columns else "sujeto_pasivo_id"
    inst_col= "institucion_id_aud"   if "institucion_id_aud"   in edges.columns else "institucion_id"

    edges["item_key"]       = edges[sp_col].astype(str).str.strip()
    edges["institution_id"] = edges[inst_col].astype(str).str.strip()
    edges["timestamp"] = pd.to_datetime(edges["timestamp"], errors="coerce")
    edges = edges.dropna(subset=["user_key","item_key","timestamp"])

    edges["__date"] = edges["timestamp"].dt.date
    edges = edges.drop_duplicates(subset=["user_key","item_key","__date"]).drop(columns="__date")

    user_index = (pd.DataFrame({"user_key": sorted(edges["user_key"].unique())})
                    .reset_index().rename(columns={"index":"user_id"}))
    item_index = (pd.DataFrame({"item_key": sorted(edges["item_key"].unique())})
                    .reset_index().rename(columns={"index":"item_id"}))

    edges = (edges.merge(user_index, on="user_key", how="left")
                  .merge(item_index, on="item_key", how="left"))

    users_df = (user_index
                .merge(active[["user_key","display_name","display_org"]].drop_duplicates("user_key"),
                       on="user_key", how="left"))

    passive = passive.rename(columns={"id":"item_key"})
    passive["item_key"] = passive["item_key"].astype(str).str.strip()
    items_df = (item_index
                .merge(passive[["item_key","nombre","cargo","institution_id"]],
                       on="item_key", how="left"))
    interactions_df = edges[["user_id","item_id","timestamp"]].sort_values(["user_id","timestamp"]).reset_index(drop=True)

    return interactions_df, users_df, items_df


In [6]:
active = pd.read_csv(os.path.join("data", "active_subjects.csv"))
aud    = pd.read_csv(os.path.join("data", "audiencies.csv"))
inst   = pd.read_csv(os.path.join("data", "institutions.csv"))
passive= pd.read_csv(os.path.join("data", "passive_subjects.csv"))
interactions_df, users_df, items_df = build_lightgcn_inputs(active, aud, passive)

/var/folders/pc/1tbslm8954q5q5cyk947n34h0000gn/T/ipykernel_90071/1667081956.py:12: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  aud["timestamp"] = pd.to_datetime(aud["fecha"], errors="coerce")


# Run trainning

In [ ]:
import itertools
import pandas as pd

# Definir hiperparámetros a probar
dim_list = [64, 128]
layers_list = [1, 2, 3]
lr_list = [5e-4, 1e-3]
l2_list = [1e-5, 1e-4]

results = []

for dim, n_layers, lr, l2 in itertools.product(dim_list, layers_list, lr_list, l2_list):
    
    print("\n===============================================")
    print(f"Probando: dim={dim}, n_layers={n_layers}, lr={lr}, l2={l2}")
    print("===============================================\n")
    
    runner = LightGCNRunner(
        dim=dim,
        n_layers=n_layers,
        lr=lr,
        l2=l2,
        batch_size=4096,
        epochs=10,         
        patience=3,
        sample_frac=0.1,
        val_metric="nDCG@10",
    )
    
    metrics = runner.fit(interactions_df)
    
    row = {
        "dim": dim,
        "n_layers": n_layers,
        "lr": lr,
        "l2": l2,
        **metrics
    }
    results.append(row)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("nDCG@10", ascending=False)
print(results_df)



In [ ]:
runner = LightGCNRunner(
    dim=128,
    n_layers=3,
    lr=1e-3,
    l2=1e-4,
    batch_size=4096,
    epochs=100,
    patience=20,
    sample_frac=1,     
)

test_metrics = runner.fit(interactions_df)
print(test_metrics)

Epoch 001 | loss 0.6907 | val Recall@10 0.0624 | val nDCG@10 0.0353 | val MAP@10 0.0270
Epoch 002 | loss 0.6702 | val Recall@10 0.0651 | val nDCG@10 0.0375 | val MAP@10 0.0291
Epoch 003 | loss 0.6217 | val Recall@10 0.0653 | val nDCG@10 0.0380 | val MAP@10 0.0297
Epoch 004 | loss 0.5515 | val Recall@10 0.0652 | val nDCG@10 0.0382 | val MAP@10 0.0300
Epoch 005 | loss 0.4732 | val Recall@10 0.0651 | val nDCG@10 0.0381 | val MAP@10 0.0299
Epoch 006 | loss 0.3982 | val Recall@10 0.0650 | val nDCG@10 0.0381 | val MAP@10 0.0299
Epoch 007 | loss 0.3292 | val Recall@10 0.0646 | val nDCG@10 0.0380 | val MAP@10 0.0298
Epoch 008 | loss 0.2739 | val Recall@10 0.0643 | val nDCG@10 0.0379 | val MAP@10 0.0298
Epoch 009 | loss 0.2285 | val Recall@10 0.0642 | val nDCG@10 0.0378 | val MAP@10 0.0297
Epoch 010 | loss 0.1937 | val Recall@10 0.0643 | val nDCG@10 0.0379 | val MAP@10 0.0298
Epoch 011 | loss 0.1644 | val Recall@10 0.0645 | val nDCG@10 0.0379 | val MAP@10 0.0298
Epoch 012 | loss 0.1419 | val Re

In [ ]:
import torch
import pandas as pd

def recommend_all_topk(runner, topk=10, batch_users=2048):
    device = runner.device
    assert runner.model is not None, "Entrena el modelo primero."

    runner.model.eval()
    with torch.no_grad():
        users_f, items_f = runner.model.propagate()
        users_f = users_f.to(device)
        items_f = items_f.to(device)

    rows = []
    for start in range(0, runner.n_users, batch_users):
        end = min(start + batch_users, runner.n_users)
        U = users_f[start:end]                      # [B, d]
        scores = U @ items_f.T                      # [B, n_items]

        for local_idx, u in enumerate(range(start, end)):
            seen = runner.tr_ui.get(u, set())
            if seen:
                seen_idx = torch.tensor(list(seen), device=device, dtype=torch.long)
                scores[local_idx, seen_idx] = -1e9

        vals, idxs = torch.topk(scores, k=topk, dim=1)   # [B, topk]
        vals = vals.detach().cpu()
        idxs = idxs.detach().cpu()

        for i, u in enumerate(range(start, end)):
            for r in range(topk):
                rows.append({
                    "user_id": u,
                    "rank": r + 1,
                    "item_id": int(idxs[i, r]),
                    "score": float(vals[i, r]),
                })

    return pd.DataFrame(rows)

# Uso:
top10_df = recommend_all_topk(runner, topk=10, batch_users=2048)
top10_df.to_csv("output/lightGCN/recommendations_top10/30-10-2025.csv", index=False)


In [29]:
import pandas as pd
import numpy as np


def recall_at_k_per_user(topk_df, te_ui, k=10):
    recalls = []
    for u, group in topk_df.groupby("user_id"):
        recommended = set(group[group["rank"] <= k]["item_id"])
        relevant = te_ui.get(u, set())
        if not relevant:
            continue
        hits = len(recommended & relevant)
        recall = hits / len(relevant)
        recalls.append({"user_id": u, "recall@10": recall, "hits": hits, "relevant": len(relevant)})
    return pd.DataFrame(recalls)

recall_df = recall_at_k_per_user(top10_df, runner.te_ui, k=10)


worst5 = recall_df.nsmallest(5, "recall@10")
best5  = recall_df.nlargest(5, "recall@10")

print("=== 5 usuarios con PEOR Recall@10 ===")
print(worst5)

print("\n=== 5 usuarios con MEJOR Recall@10 ===")
print(best5)


=== 5 usuarios con PEOR Recall@10 ===
   user_id  recall@10  hits  relevant
0        0        0.0     0         1
1        1        0.0     0         1
2        2        0.0     0         1
3        3        0.0     0         1
4        4        0.0     0         1

=== 5 usuarios con MEJOR Recall@10 ===
     user_id  recall@10  hits  relevant
22        22        1.0     1         1
156      156        1.0     1         1
217      217        1.0     1         1
226      226        1.0     1         1
271      271        1.0     1         1


In [ ]:
import torch
import json
import os

save_dir = "output/lightGCN/models"
os.makedirs(save_dir, exist_ok=True)

# --- 1. Guarda los pesos del modelo ---
torch.save(runner.model.state_dict(), f"{save_dir}/lightgcn_weights.pt")

# --- 2. Guarda la configuración y metadatos (opcional pero muy útil) ---
config = {
    "n_users": runner.n_users,
    "n_items": runner.n_items,
    "dim": runner.dim,
    "n_layers": runner.n_layers,
    "device": runner.device,
}
with open(f"{save_dir}/lightgcn_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✅ Modelo y configuración guardados en", save_dir)


✅ Modelo y configuración guardados en output/lightGCN/models


In [12]:
import torch
import json
from src.state_of_art_models.lightGCN_base import LightGCN

# --- 1. Carga la configuración ---
with open("output/lightGCN/models/lightgcn_config.json") as f:
    cfg = json.load(f)

# --- 2. Reconstruye el modelo ---
# A_hat no es necesario para inferencia (solo si quieres seguir entrenando)
dummy_A_hat = torch.sparse_coo_tensor(
    torch.zeros((2, 0), dtype=torch.long), torch.zeros(0), size=(cfg["n_users"] + cfg["n_items"], cfg["n_users"] + cfg["n_items"])
).coalesce()

model = LightGCN(
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    dim=cfg["dim"],
    n_layers=cfg["n_layers"],
    A_hat=dummy_A_hat,
)

# --- 3. Carga los pesos ---
model.load_state_dict(torch.load("output/lightGCN/models/lightgcn_weights.pt", map_location=cfg["device"]))
model.eval()
print("✅ Modelo cargado correctamente")

# Ejemplo de uso (usando tu runner para consistencia)
runner.model = model.to(cfg["device"])


✅ Modelo cargado correctamente


In [13]:
# Propaga embeddings
users_f, items_f = model.propagate()

# Escoge un usuario
u = 42  # por ejemplo
scores = (users_f[u] * items_f).sum(dim=1)

# (opcional) enmascara ítems vistos
seen = runner.tr_ui.get(u, set()) if hasattr(runner, "tr_ui") else set()
if seen:
    scores[list(seen)] = -1e9

# Obtén los top-10 ítems más recomendados
top_items = torch.topk(scores, 10).indices.cpu().tolist()
print("Recomendaciones para usuario", u, ":", top_items)


Recomendaciones para usuario 42 : [1192, 5022, 2587, 9879, 10942, 10412, 260, 16610, 3647, 6035]


In [ ]:


top10_df = pd.read_csv("output/lightGCN/recommendations_top10/30-10-2025.csv")

In [ ]:
import pandas as pd
import numpy as np

top10_df = pd.read_csv("output/lightGCN/recommendations_top10/30-10-2025.csv")

item_counts = top10_df["item_id"].value_counts().sort_index().values

def gini(array):
    array = np.array(array, dtype=np.float64)
    if np.amin(array) < 0:
        array = array - np.min(array)
    if np.sum(array) == 0:
        return 0.0
    sorted_arr = np.sort(array)
    n = len(array)
    cum = np.cumsum(sorted_arr)
    g = (n + 1 - 2 * np.sum(cum) / cum[-1]) / n
    return g

gini_top10 = gini(item_counts)

print("Índice Gini de exposición (top-10):", gini_top10)


Índice Gini de exposición (top-10): 0.8956611027140478
